In [1]:
# Cell 1: Load libraries, data and trained models for LIME
# LIME = Local Interpretable Model-Agnostic Explanations
# LIME explains ONE prediction at a time (local explanation)
# It works by creating small changes around one sample
# then trains a simple model to explain the complex model's behavior

import numpy as np
import pandas as pd
import lime
import lime.lime_tabular  # tabular = for CSV/table data (not images or text)
import matplotlib
matplotlib.use('Agg')  # prevents display errors on Mac
import matplotlib.pyplot as plt
import json
import time

# path where all processed files and models are saved
save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAV-Cyber-Physical/processed/"

# load test and train data
# LIME needs X_train as background reference data
# LIME needs X_test to explain predictions on new samples
X_train = np.load(save_path + "X_train.npy")
X_test  = np.load(save_path + "X_test.npy")
y_test  = pd.read_csv(save_path + "y_test.csv").squeeze()

# load class names and feature names
le_classes    = pd.read_csv(save_path + "label_classes.csv").squeeze().tolist()
feature_names = pd.read_csv(save_path + "feature_names.csv").squeeze().tolist()

# load trained models
import joblib
rf_model = joblib.load(save_path + "rf_model.joblib")
print("RF model loaded!")

from tensorflow import keras
cnn_model = keras.models.load_model(save_path + "cnn_model.keras")
ae_model  = keras.models.load_model(save_path + "ae_model.keras")
print("CNN model loaded!")
print("AE model loaded!")

# load threshold for autoencoder
ae_threshold = np.load(save_path + "ae_threshold.npy")[0]
print(f"AE threshold: {ae_threshold:.6f}")

# load benign label index
# needed to filter benign samples for AE evaluation
benign_label = le_classes.index('benign')
print(f"Benign label index: {benign_label}")

print(f"\nX_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Classes: {le_classes}")
print(f"Features: {len(feature_names)}")
print("\nAll loaded!")

RF model loaded!


2026-05-11 14:00:51.205473: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


CNN model loaded!
AE model loaded!
AE threshold: 2.250115
Benign label index: 2

X_train: (23171, 37)
X_test:  (9931, 37)
Classes: ['DoS attack', 'Replay', 'benign']
Features: 37

All loaded!


In [7]:
# Cell 2: Apply LIME to Random Forest
print("Running LIME on Random Forest...")

explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train,
    feature_names=feature_names,
    class_names=le_classes,
    mode='classification'
)

sample_size = 50
np.random.seed(42)
sample_idx = np.random.choice(len(X_test), sample_size, replace=False)

start = time.time()
lime_rf_global = np.zeros(len(feature_names))

for i, idx in enumerate(sample_idx):
    exp = explainer.explain_instance(
        X_test[idx],
        rf_model.predict_proba,
        num_features=len(feature_names)
    )
    # fix: get first available key instead of hardcoded 0
    map_result = exp.as_map()
    first_key = list(map_result.keys())[0]
    for feat_idx, weight in map_result[first_key]:
        lime_rf_global[feat_idx] += abs(weight)

    if (i+1) % 10 == 0:
        print(f"  Progress: {i+1}/{sample_size} samples done...")

lime_rf_global /= sample_size
elapsed_rf = round(time.time() - start, 2)
print(f"LIME RF complete! Time: {elapsed_rf}s")

top_idx_rf = [int(i) for i in np.argsort(lime_rf_global)[::-1][:10]]

print(f"\nTop 10 Features — RF LIME:")
print(f"{'Rank':<6} {'Feature':<35} {'LIME Weight':<15}")
print("-" * 56)
for i, idx in enumerate(top_idx_rf):
    print(f"  {i+1:<4} {feature_names[idx]:<35} {lime_rf_global[idx]:.6f}")

plt.figure(figsize=(10, 7))
plt.barh(
    [feature_names[i] for i in top_idx_rf[::-1]],
    [lime_rf_global[i] for i in top_idx_rf[::-1]],
    color='steelblue'
)
plt.xlabel('Mean |LIME Weight|')
plt.title('LIME Global Feature Importance — Random Forest (UAV Dataset)')
plt.tight_layout()
plt.savefig(save_path + "lime_rf_global.png", dpi=150, bbox_inches='tight')
plt.show()
print("RF LIME plot saved!")

Running LIME on Random Forest...
  Progress: 10/50 samples done...
  Progress: 20/50 samples done...
  Progress: 30/50 samples done...
  Progress: 40/50 samples done...
  Progress: 50/50 samples done...
LIME RF complete! Time: 8.16s

Top 10 Features — RF LIME:
Rank   Feature                             LIME Weight    
--------------------------------------------------------
  1    timestamp_c                         0.077666
  2    frame.number                        0.051473
  3    wlan.fc.type                        0.040045
  4    wlan.seq                            0.036681
  5    wlan.duration                       0.035024
  6    wlan.fc.subtype                     0.019742
  7    ip.flags                            0.015207
  8    data.data                           0.013529
  9    wlan.ta                             0.012689
  10   wlan.sa                             0.012626
RF LIME plot saved!


/var/folders/28/cggd8l710jz37nj7nfdl032r0000gn/T/ipykernel_1888/965432481.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# Cell 3: Apply LIME to 1D-CNN
print("Running LIME on 1D-CNN...")

def cnn_predict_proba(X):
    X_3d = X.reshape(X.shape[0], X.shape[1], 1)
    return cnn_model.predict(X_3d, verbose=0)

start = time.time()
lime_cnn_global = np.zeros(len(feature_names))

for i, idx in enumerate(sample_idx):
    exp = explainer.explain_instance(
        X_test[idx],
        cnn_predict_proba,
        num_features=len(feature_names)
    )
    # fix: get first available key
    map_result = exp.as_map()
    first_key = list(map_result.keys())[0]
    for feat_idx, weight in map_result[first_key]:
        lime_cnn_global[feat_idx] += abs(weight)

    if (i+1) % 10 == 0:
        print(f"  Progress: {i+1}/{sample_size} samples done...")

lime_cnn_global /= sample_size
elapsed_cnn = round(time.time() - start, 2)
print(f"LIME CNN complete! Time: {elapsed_cnn}s")

top_idx_cnn = [int(i) for i in np.argsort(lime_cnn_global)[::-1][:10]]

print(f"\nTop 10 Features — CNN LIME:")
print(f"{'Rank':<6} {'Feature':<35} {'LIME Weight':<15}")
print("-" * 56)
for i, idx in enumerate(top_idx_cnn):
    print(f"  {i+1:<4} {feature_names[idx]:<35} {lime_cnn_global[idx]:.6f}")

plt.figure(figsize=(10, 7))
plt.barh(
    [feature_names[i] for i in top_idx_cnn[::-1]],
    [lime_cnn_global[i] for i in top_idx_cnn[::-1]],
    color='steelblue'
)
plt.xlabel('Mean |LIME Weight|')
plt.title('LIME Global Feature Importance — 1D-CNN (UAV Dataset)')
plt.tight_layout()
plt.savefig(save_path + "lime_cnn_global.png", dpi=150, bbox_inches='tight')
plt.show()
print("CNN LIME plot saved!")

Running LIME on 1D-CNN...
  Progress: 10/50 samples done...
  Progress: 20/50 samples done...
  Progress: 30/50 samples done...
  Progress: 40/50 samples done...
  Progress: 50/50 samples done...
LIME CNN complete! Time: 34.2s

Top 10 Features — CNN LIME:
Rank   Feature                             LIME Weight    
--------------------------------------------------------
  1    timestamp_c                         0.203351
  2    frame.len                           0.141898
  3    wlan.duration                       0.127906
  4    wlan.seq                            0.111319
  5    frame.number                        0.080728
  6    frame.protocols                     0.072048
  7    tcp.srcport                         0.061860
  8    llc.type                            0.059086
  9    wlan.fc.subtype                     0.058641
  10   data.data                           0.049210
CNN LIME plot saved!


/var/folders/28/cggd8l710jz37nj7nfdl032r0000gn/T/ipykernel_1888/1413832839.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# Cell 4: Apply LIME to Autoencoder - FIXED
print("Running LIME on Autoencoder...")

explainer_ae = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train,
    feature_names=feature_names,
    mode='regression'
)

def ae_predict_mse(X):
    X_reconstructed = ae_model.predict(X, verbose=0)
    mse = np.mean(np.power(X - X_reconstructed, 2), axis=1)
    return mse

start = time.time()
lime_ae_global = np.zeros(len(feature_names))

for i, idx in enumerate(sample_idx):
    exp = explainer_ae.explain_instance(
        X_test[idx],
        ae_predict_mse,
        num_features=len(feature_names)
    )
    # use as_map()[0] directly — returns feature INDEX not name
    # key 0 = positive class weights
    map_result = exp.as_map()
    first_key = list(map_result.keys())[0]
    for feat_idx, weight in map_result[first_key]:
        if feat_idx < len(feature_names):
            lime_ae_global[feat_idx] += abs(weight)

    if (i+1) % 10 == 0:
        print(f"  Progress: {i+1}/{sample_size} samples done...")

lime_ae_global /= sample_size
elapsed_ae = round(time.time() - start, 2)
print(f"LIME AE complete! Time: {elapsed_ae}s")

top_idx_ae = [int(i) for i in np.argsort(lime_ae_global)[::-1][:10]]

print(f"\nTop 10 Features — AE LIME:")
print(f"{'Rank':<6} {'Feature':<35} {'LIME Weight':<15}")
print("-" * 56)
for i, idx in enumerate(top_idx_ae):
    print(f"  {i+1:<4} {feature_names[idx]:<35} {lime_ae_global[idx]:.6f}")

plt.figure(figsize=(10, 7))
plt.barh(
    [feature_names[i] for i in top_idx_ae[::-1]],
    [lime_ae_global[i] for i in top_idx_ae[::-1]],
    color='steelblue'
)
plt.xlabel('Mean |LIME Weight|')
plt.title('LIME Global Feature Importance — Autoencoder (UAV Dataset)')
plt.tight_layout()
plt.savefig(save_path + "lime_ae_global.png", dpi=150, bbox_inches='tight')
plt.show()
print("AE LIME plot saved!")

Running LIME on Autoencoder...
  Progress: 10/50 samples done...
  Progress: 20/50 samples done...
  Progress: 30/50 samples done...
  Progress: 40/50 samples done...
  Progress: 50/50 samples done...
LIME AE complete! Time: 24.5s

Top 10 Features — AE LIME:
Rank   Feature                             LIME Weight    
--------------------------------------------------------
  1    tcp.dstport                         0.386338
  2    tcp.hdr_len                         0.348034
  3    tcp.options                         0.340256
  4    tcp.window_size                     0.327043
  5    ip.flags                            0.325845
  6    tcp.flags                           0.325772
  7    tcp.srcport                         0.314869
  8    tcp.seq_raw                         0.314541
  9    udp.length                          0.096721
  10   data.data                           0.085975
AE LIME plot saved!


/var/folders/28/cggd8l710jz37nj7nfdl032r0000gn/T/ipykernel_1888/2616714732.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
# Cell 5: Save all LIME results to JSON

# save RF LIME results
lime_rf_results = {
    "model": "RandomForest",
    "dataset": "UAV-Cyber-Physical",
    "xai_method": "LIME",
    "sample_size": sample_size,
    "time_seconds": elapsed_rf,
    "top_10_features": {
        feature_names[int(i)]: round(float(lime_rf_global[int(i)]), 6)
        for i in top_idx_rf
    }
}
with open(save_path + "lime_rf_results.json", "w") as f:
    json.dump(lime_rf_results, f, indent=2)
print("RF LIME JSON saved!")

# save CNN LIME results
lime_cnn_results = {
    "model": "1D-CNN",
    "dataset": "UAV-Cyber-Physical",
    "xai_method": "LIME",
    "sample_size": sample_size,
    "time_seconds": elapsed_cnn,
    "top_10_features": {
        feature_names[int(i)]: round(float(lime_cnn_global[int(i)]), 6)
        for i in top_idx_cnn
    }
}
with open(save_path + "lime_cnn_results.json", "w") as f:
    json.dump(lime_cnn_results, f, indent=2)
print("CNN LIME JSON saved!")

# save AE LIME results
lime_ae_results = {
    "model": "Autoencoder",
    "dataset": "UAV-Cyber-Physical",
    "xai_method": "LIME",
    "sample_size": sample_size,
    "time_seconds": elapsed_ae,
    "top_10_features": {
        feature_names[int(i)]: round(float(lime_ae_global[int(i)]), 6)
        for i in top_idx_ae
    }
}
with open(save_path + "lime_ae_results.json", "w") as f:
    json.dump(lime_ae_results, f, indent=2)
print("AE LIME JSON saved!")

print("\nAll LIME results saved!")
print(f"\nSummary:")
print(f"  RF  LIME #1: {feature_names[top_idx_rf[0]]}")
print(f"  CNN LIME #1: {feature_names[top_idx_cnn[0]]}")
print(f"  AE  LIME #1: {feature_names[top_idx_ae[0]]}")

RF LIME JSON saved!
CNN LIME JSON saved!
AE LIME JSON saved!

All LIME results saved!

Summary:
  RF  LIME #1: timestamp_c
  CNN LIME #1: timestamp_c
  AE  LIME #1: tcp.dstport


In [11]:
# Debug cell: check what as_list() actually returns
exp_test = explainer_ae.explain_instance(
    X_test[0],
    ae_predict_mse,
    num_features=5
)
print("as_list() output:")
for item in exp_test.as_list():
    print(f"  {item}")

print("\nas_map() output:")
print(exp_test.as_map())

as_list() output:
  ('tcp.dstport <= -0.27', -0.3927369230767345)
  ('tcp.options <= -0.27', -0.351441028026091)
  ('tcp.hdr_len <= -0.27', -0.3430542608690685)
  ('tcp.window_size <= -0.27', -0.3348175758070149)
  ('tcp.seq_raw <= -0.23', -0.3287623137494604)

as_map() output:
{0: [(22, 0.3927369230767345), (28, 0.351441028026091), (25, 0.3430542608690685), (27, 0.3348175758070149), (23, 0.3287623137494604)], 1: [(22, -0.3927369230767345), (28, -0.351441028026091), (25, -0.3430542608690685), (27, -0.3348175758070149), (23, -0.3287623137494604)]}
